# Fine-Tuning LayoutLMv3 on FATURA2 Invoices (5,000 Subset)
**Intelligent Document Processing Pipeline**

This notebook fine-tunes `microsoft/layoutlmv3-base` on `mathieu1256/FATURA2-invoices` for token classification and structured invoice information extraction.

### Pipeline Overview:
- **Dataset**: `mathieu1256/FATURA2-invoices` (synthetic invoices across 50 templates)
- **Training Subset**: Exactly 5,000 examples from the 8,600 training split
  - 4,400 training examples (22 complete templates, 0% template overlap)
  - 600 validation examples (3 complete templates, 0% template overlap)
  - Official 1,400-example test split kept untouched for holdout evaluation (7 complete templates)
- **Strict Anti-Leakage Strategy**: Enforces 200-sample template block alignment to guarantee ZERO template overlap between train and val
- **Training Configuration**: 3 epochs, mixed precision (`fp16`), batch size 4 (gradient accumulation 2 = effective batch size 8), Early Stopping
- **Target Initial Duration**: Benchmark estimated at ~25-45 minutes on Google Colab T4 GPU
- **Export**: Checkpoint saved to Google Drive and/or downloadable zip for backend integration


In [ ]:
# 1. Environment & Hardware Verification
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("WARNING: No GPU detected. In Colab, go to Runtime > Change runtime type > Select T4 GPU.")


In [ ]:
# 2. Install Required Dependencies
!pip install -q --upgrade transformers datasets seqeval evaluate accelerate torchvision Pillow pytesseract


### 3. Optional: Mount Google Drive
Mount Google Drive so the fine-tuned checkpoint is automatically backed up to your personal Drive storage.


In [ ]:
# 3. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    DRIVE_DIR = '/content/drive/MyDrive/idp_checkpoints/layoutlmv3-fatura2'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Mounted Google Drive. Output directory: {DRIVE_DIR}")
except Exception as e:
    DRIVE_MOUNTED = False
    print("Google Drive not mounted or running locally. Checkpoints will be stored locally.")


### 4. Load FATURA2 Dataset & Zero-Leakage Template Partitioning
In FATURA (arXiv:2311.11856), 50 templates generate exactly 200 invoices each. To guarantee **0.0% template overlap** across splits, sample counts align with strict 200-sample template boundaries:
- **Training split**: 4,400 examples (22 complete templates: Templates 0..21)
- **Validation split**: 600 examples (3 complete templates: Templates 22..24)
- **Holdout test split**: 1,400 examples (7 complete templates: Templates 43..49, completely untouched)
- **Template overlap**: **0.0%** (zero template leakage)


In [ ]:
# 4. Load Dataset and Partition
from datasets import load_dataset

print("Downloading and loading mathieu1256/FATURA2-invoices...")
dataset = load_dataset("mathieu1256/FATURA2-invoices")
print("Dataset structure:", dataset)

if "test" in dataset:
    raw_train = dataset["train"]
    raw_test = dataset["test"]
else:
    print("Single split detected on Hub. Slicing official 8,600 train and 1,400 holdout test...")
    total_len = len(dataset["train"])
    train_end = min(8600, total_len)
    raw_train = dataset["train"].select(range(0, train_end))
    raw_test = dataset["train"].select(range(train_end, total_len))

train_samples = 4400
val_samples = 600

# Strict zero-leakage partition strategy
cols = raw_train.column_names
template_col = next((c for c in ["template_id", "template", "template_name", "layout"] if c in cols), None)

if template_col:
    print(f"Found template column: '{template_col}'. Partitioning by unique templates...")
    unique_templates = sorted(list(set(raw_train[template_col])))
    val_count = max(1, round(len(unique_templates) * (val_samples / (train_samples + val_samples))))
    val_templates = set(unique_templates[-val_count:])
    train_templates = set(unique_templates[:-val_count])
    
    train_indices = [i for i, t in enumerate(raw_train[template_col]) if t in train_templates][:train_samples]
    val_indices = [i for i, t in enumerate(raw_train[template_col]) if t in val_templates][:val_samples]
else:
    template_block_size = 200
    print(f"Partitioning strict 200-sample template blocks (Train: 0..{train_samples} [Templates 0..{train_samples // template_block_size - 1}], Val: {train_samples}..{train_samples + val_samples} [Templates {train_samples // template_block_size}..{(train_samples + val_samples) // template_block_size - 1}])... Template Overlap: 0.0%")
    train_indices = list(range(0, train_samples))
    val_indices = list(range(train_samples, train_samples + val_samples))

train_split = raw_train.select(train_indices)
val_split = raw_train.select(val_indices)

print(f"Training split:   {len(train_split)} examples (22 full templates)")
print(f"Validation split: {len(val_split)} examples (3 full templates)")
print(f"Test split:       {len(raw_test)} examples (7 full templates, official holdout)")
print("Template Overlap between Train and Validation: 0.0% (Zero Leakage)")


### 5. Inspect Dataset Labels and BIO Mapping (Verified Zenodo Mapping)
We establish the exact ground-truth mapping between integer IDs in `ner_tags` and semantic categories by verifying against the official Zenodo FATURA dataset annotations (https://zenodo.org/records/10371464).

**Zenodo Annotation Investigation Results:**
- In `mathieu1256/FATURA2-invoices`, `ner_tags` contains integer IDs **0–13**, with **ID 7 completely absent** across the entire 10,000-sample dataset.
- The FATURA authors generated `layoutlm_HF_format` from the original annotations by mapping primary KIE entities to specific IDs (0–12) and collapsing background/secondary elements into **ID 13 (`OTHER`)**.
- **Verified Integer Tag Mapping:**
  - `0`: `SELLER_SITE` (Vendor website URL)
  - `1`: `TOTAL` (Invoice grand total amount)
  - `2`: `NOTE` (Total in words / footnote)
  - `3`: `DATE` (Invoice date)
  - `4`: `DUE_DATE` (Payment due date)
  - `5`: `BILL_TO` (Buyer / customer name and billing block)
  - `6`: `SELLER` (Seller / vendor name)
  - `7`: `OTHER` (Unused / absent ID across dataset)
  - `8`: `BILL_TO` (Explicit BILL_TO section)
  - `9`: `SEND_TO` (Explicit SHIP_TO / SEND_TO destination)
  - `10`: `TABLE` (Product line-item table structure)
  - `11`: `LOGO` (Template logo graphic)
  - `12`: `NUMBER` (Invoice number / identifier)
  - `13`: `OTHER` (Outside / unclassified background text elements)
- **Confirmation:** `Buyer` (Tag 5 and Tag 8) is mapped strictly to `BILL_TO`, completely distinct from `DUE_DATE` (Tag 4).

In [ ]:
# 5. Label Discovery and BIO Tag Setup (Zenodo Ground-Truth Verified)
# Official integer ID mapping verified against Zenodo annotation files (https://zenodo.org/records/10371464)
FATURA2_INTEGER_TAG_MAP = {
    0: 'SELLER_SITE',
    1: 'TOTAL',
    2: 'NOTE',            # Total in words / notes
    3: 'DATE',             # Invoice date
    4: 'DUE_DATE',         # Payment due date
    5: 'BILL_TO',          # Buyer / customer name & billing block
    6: 'SELLER',           # Seller / vendor name
    7: 'OTHER',            # Absent across all dataset records (mapped to outside)
    8: 'BILL_TO',          # Explicit BILL_TO block
    9: 'SEND_TO',          # Explicit SHIP_TO / SEND_TO block
    10: 'TABLE',           # Product line-item table
    11: 'LOGO',            # Vendor logo
    12: 'NUMBER',          # Invoice number / ID
    13: 'OTHER',           # Background / non-target elements
}

# Full 24 semantic classes from Table 2 of FATURA (arXiv:2311.11856)
FATURA2_24_CATEGORIES = [
    'TABLE', 'LOGO', 'TITLE', 'NUMBER', 'DATE', 'DUE_DATE',
    'PO_NUMBER', 'CONDITIONS', 'SELLER', 'SELLER_ADDRESS', 'SELLER_SITE',
    'BILL_TO', 'SEND_TO', 'SUBTOTAL', 'TAX', 'DISCOUNT', 'AMOUNT_DUE',
    'TOTAL', 'ITEM_DESCRIPTION', 'ITEM_QUANTITY', 'ITEM_UNIT_PRICE',
    'ITEM_TOTAL', 'NOTE',
]

# Aliases for string-based tags
CATEGORY_ALIASES = {
    'BUYER': 'BILL_TO', 'BUYER_NAME': 'BILL_TO', 'CUSTOMER': 'BILL_TO', 'CUSTOMER_NAME': 'BILL_TO',
    'BUYER_ADDRESS': 'SEND_TO', 'CUSTOMER_ADDRESS': 'SEND_TO', 'SHIPPING_ADDRESS': 'SEND_TO', 'SHIP_TO': 'SEND_TO',
    'INVOICE_NUMBER': 'NUMBER', 'INV_NO': 'NUMBER', 'INVOICE_ID': 'NUMBER', 'INVOICE_DATE': 'DATE',
    'GRAND_TOTAL': 'TOTAL', 'TOTAL_PRICE': 'TOTAL', 'SUB_TOTAL': 'SUBTOTAL', 'TAXABLE_AMOUNT': 'SUBTOTAL',
    'TAX_AMOUNT': 'TAX', 'TOTAL_TAX': 'TAX', 'VAT': 'TAX', 'GST': 'TAX',
    'VENDOR': 'SELLER', 'VENDOR_NAME': 'SELLER', 'SUPPLIER': 'SELLER',
    'REMARKS': 'NOTE', 'TERMS': 'CONDITIONS', 'WEBSITE': 'SELLER_SITE',
}

def build_bio_labels():
    '''Build canonical BIO label list for FATURA2 token classification.'''
    labels = ['O']
    for cat in FATURA2_24_CATEGORIES:
        labels.append(f'B-{cat}')
        labels.append(f'I-{cat}')
    return labels

def convert_spans_to_bio(
    raw_tags,
    words,
    boxes,
    bio_label2id,
):
    '''Convert raw token tags into aligned BIO tag names and integer IDs using verified Zenodo mapping.'''
    bio_tags = []
    bio_ids = []
    prev_cat = None
    prev_box = None

    for idx, raw_t in enumerate(raw_tags):
        if isinstance(raw_t, int):
            cat_name = FATURA2_INTEGER_TAG_MAP.get(raw_t, 'OTHER')
        elif isinstance(raw_t, str):
            raw_str = raw_t.strip().upper()
            cat_name = CATEGORY_ALIASES.get(raw_str, raw_str)
        else:
            cat_name = 'OTHER'

        if cat_name in ('O', 'OTHER', 'BACKGROUND', 'NONE', '0'):
            bio_tag = 'O'
            prev_cat = None
            prev_box = None
        else:
            box = boxes[idx] if boxes and idx < len(boxes) else None
            is_continuation = (prev_cat == cat_name)
            if is_continuation and box and prev_box and len(box) == 4 and len(prev_box) == 4:
                prev_y_center = (prev_box[1] + prev_box[3]) / 2.0
                curr_y_center = (box[1] + box[3]) / 2.0
                y_diff = abs(curr_y_center - prev_y_center)
                carriage_return = box[0] < prev_box[0] - 100
                if y_diff > 35 or (carriage_return and y_diff > 15):
                    is_continuation = False

            bio_tag = f'I-{cat_name}' if is_continuation else f'B-{cat_name}'
            prev_cat = cat_name
            prev_box = box

        if bio_tag not in bio_label2id:
            bio_tag = 'O'

        bio_tags.append(bio_tag)
        bio_ids.append(bio_label2id[bio_tag])

    return bio_tags, bio_ids

labels = build_bio_labels()
id2label = {i: lbl for i, lbl in enumerate(labels)}
label2id = {lbl: i for i, lbl in enumerate(labels)}
num_labels = len(labels)

print(f'Total Canonical BIO classes: {num_labels}')
print(f'First 10 BIO labels: {labels[:10]}')
print('Verified Zenodo Integer Tag Map:')
for tid, tname in sorted(FATURA2_INTEGER_TAG_MAP.items()):
    print(f'  Tag {tid:2d} -> {tname}')


### 6. Verify Preprocessing on Sample Record
We verify sample dimensions, bounding boxes, and label alignment before running batched encoding.


In [ ]:
# 6. Sample Record Verification (Verified Zenodo Mapping)
sample = train_split[0]
sample_img = sample['image'].convert('RGB')
sample_words = sample.get('words', sample.get('tokens', []))
sample_boxes = sample.get('bboxes', sample.get('boxes', []))
sample_tags = sample.get('ner_tags', [])

print(f'Sample Image Size: {sample_img.size}')
print(f'Word count: {len(sample_words)}')
print('First 8 words:   ', sample_words[:8])
print('First 8 boxes:   ', sample_boxes[:8])
print('First 8 raw tags:', sample_tags[:8])

# Test conversion on sample record
sample_bio_tags, sample_bio_ids = convert_spans_to_bio(
    raw_tags=sample_tags,
    words=sample_words,
    boxes=sample_boxes,
    bio_label2id=label2id,
)
print('First 8 converted BIO tags:', sample_bio_tags[:8])
print('First 8 converted BIO IDs: ', sample_bio_ids[:8])
for k in range(min(8, len(sample_words))):
    print(f'  Token {k}: "{sample_words[k]}" (RawTag={sample_tags[k]}) -> {sample_bio_tags[k]} (ID={sample_bio_ids[k]})')


### 7. LayoutLMv3 Preprocessing Pipeline with Verified Zenodo BIO Converter
We configure `LayoutLMv3Processor` with `apply_ocr=False`. Word-level bounding boxes are scaled to the `0-1000` coordinate space.

**Pre-Tokenization Alignment Guarantee:**
- Maps raw integer tags (0–13) through `FATURA2_INTEGER_TAG_MAP` to authentic semantic entities.
- Converts consecutive entity tokens into `B-<CAT>` on start and `I-<CAT>` on continuation.
- Maps tags directly into canonical integer IDs via `label2id`, eliminating tag scrambling.
- Subword continuations and special tokens (`<s>`, `</s>`, `<pad>`) are mapped to `-100` for PyTorch cross-entropy loss.

In [ ]:
# 7. Preprocessing Implementation
from transformers import AutoProcessor

model_checkpoint = 'microsoft/layoutlmv3-base'
processor = AutoProcessor.from_pretrained(model_checkpoint, apply_ocr=False)

def normalize_bbox(box, width, height):
    if not box or len(box) != 4:
        return [0, 0, 0, 0]
    x0, y0, x1, y1 = [int(v) for v in box]
    w, h = max(int(width), 1), max(int(height), 1)
    if max(x0, x1) <= 1000 and max(y0, y1) <= 1000 and w > 1000 and h > 1000:
        return [max(0, min(1000, x0)), max(0, min(1000, y0)), max(x0, min(1000, x1)), max(y0, min(1000, y1))]
    nx0 = max(0, min(1000, int(x0 * 1000 / w)))
    ny0 = max(0, min(1000, int(y0 * 1000 / h)))
    nx1 = max(nx0, min(1000, int(x1 * 1000 / w)))
    ny1 = max(ny0, min(1000, int(y1 * 1000 / h)))
    return [nx0, ny0, nx1, ny1]

def prepare_fatura2_batch(batch):
    batch_images = []
    batch_words = []
    batch_boxes = []
    batch_labels = []

    for i in range(len(batch['image'])):
        img = batch['image'][i].convert('RGB')
        w, h = img.size
        words = batch.get('words', [[]])[i] if 'words' in batch else batch.get('tokens', [[]])[i]
        boxes = batch.get('bboxes', [[]])[i] if 'bboxes' in batch else batch.get('boxes', [[]])[i]
        raw_tags = batch.get('ner_tags', [[]])[i]

        norm_boxes = [normalize_bbox(b, w, h) for b in boxes]
        if not words:
            words = ['empty']
            norm_boxes = [[0, 0, 10, 10]]
            raw_tags = [13]

        _, tag_ids = convert_spans_to_bio(
            raw_tags=raw_tags,
            words=words,
            boxes=norm_boxes,
            bio_label2id=label2id,
        )

        batch_images.append(img)
        batch_words.append(words)
        batch_boxes.append(norm_boxes)
        batch_labels.append(tag_ids)

    return processor(
        batch_images,
        batch_words,
        boxes=batch_boxes,
        word_labels=batch_labels,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt',
    )

print('Tokenizing and encoding datasets (batched)... This takes ~1-2 minutes on Colab.')
encoded_train = train_split.map(prepare_fatura2_batch, batched=True, batch_size=4, remove_columns=train_split.column_names)
encoded_val = val_split.map(prepare_fatura2_batch, batched=True, batch_size=4, remove_columns=val_split.column_names)
encoded_test = raw_test.map(prepare_fatura2_batch, batched=True, batch_size=4, remove_columns=raw_test.column_names)
print('Datasets successfully encoded!')


### 8. Model Setup & Seqeval Evaluation Metrics
We load `microsoft/layoutlmv3-base` with the token classification head and compute precision, recall, and F1 score with seqeval.


In [ ]:
# 8. Model and Seqeval Metrics Setup
import numpy as np
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    predictions, labels_batch = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []
    for pred_row, label_row in zip(predictions, labels_batch):
        pred_tags = []
        label_tags = []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                pred_tags.append(id2label.get(p, "O"))
                label_tags.append(id2label.get(l, "O"))
        true_preds.append(pred_tags)
        true_labels.append(label_tags)

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


### 9. Training Execution (3 Epochs, Mixed Precision)
We run 3 epochs of fine-tuning with:
- **Batch size**: 4 (with gradient accumulation of 2 = effective batch size of 8)
- **Mixed precision**: `fp16=True` on CUDA
- **Evaluation strategy**: Per epoch on the held-out validation set
- **Model selection**: Top 2 checkpoints preserved, best checkpoint automatically reloaded
- **Early stopping**: Patience of 2 epochs


In [ ]:
# 9. Trainer Execution
output_dir = "./checkpoints/layoutlmv3-fatura2"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("Starting LayoutLMv3 training (3 epochs, effective batch size = 8)...\n")
train_result = trainer.train()
print("\nTraining complete! Metrics:", train_result.metrics)


### 10. Checkpoint Evaluation on Validation & Test Sets
We evaluate the best model checkpoint on the validation set and the 1,400-example official test set.


In [ ]:
# 10. Checkpoint Evaluation
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("--- Validation Set Results (600 held-out samples, 3 templates) ---")
val_results = trainer.evaluate(encoded_val)
for k, v in val_results.items():
    print(f"  {k}: {v}")

print("\n--- Official Test Set Results (1,400 holdout invoices) ---")
test_results = trainer.evaluate(encoded_test)
for k, v in test_results.items():
    print(f"  {k}: {v}")


### 11. Checkpoint Export (Google Drive & Zip Download)
We package the fine-tuned model weights, processor, and metadata into a zip archive and export it to Google Drive or trigger a browser download.


In [ ]:
# 11. Packaging and Export
import shutil
import json

summary = {
    "dataset": "mathieu1256/FATURA2-invoices",
    "train_samples": len(train_split),
    "val_samples": len(val_split),
    "test_samples": len(raw_test),
    "num_labels": num_labels,
    "epochs": 3,
    "train_metrics": train_result.metrics,
    "val_metrics": val_results,
    "test_metrics": test_results,
}
with open(f"{output_dir}/training_results.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_filename = "layoutlmv3-fatura2-final"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"Created archive: {zip_filename}.zip")

if DRIVE_MOUNTED:
    dest = f"{DRIVE_DIR}/layoutlmv3-fatura2-final.zip"
    shutil.copy(f"{zip_filename}.zip", dest)
    print(f"Copied archive to Google Drive: {dest}")
else:
    try:
        from google.colab import files
        files.download(f"{zip_filename}.zip")
        print("Download prompted in browser.")
    except Exception:
        print(f"Archive saved locally: {os.path.abspath(zip_filename + '.zip')}")


### Backend Integration Instructions
To use this newly fine-tuned model in your Intelligent Document Processing application:

1. Extract the downloaded `layoutlmv3-fatura2-final.zip` into:
   `c:\Users\megha\intelligent-document-processing\backend\checkpoints\layoutlmv3-fatura2`
2. In `backend/.env`, set the model path:
   ```env
   IDP_LAYOUTLM_MODEL=checkpoints/layoutlmv3-fatura2
   ```
3. Restart the backend server:
   ```bash
   cd backend
   python -m app.main
   ```
4. The backend will automatically recognize the FATURA2 entity labels in `id2label` and switch from generic FUNSD spatial pairing to direct invoice entity token classification.
